In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models
from transformers import VisualBertModel, BertTokenizer, VisualBertConfig
from sklearn.metrics import classification_report
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

train_path = "/kaggle/input/sliver-multi/silver_multi.json"
val_path = "/kaggle/input/data-sarcasm-clip/multi_dev.json"
test_path = "/kaggle/input/data-sarcasm-clip/multi_test.json"
img_dir = "/kaggle/input/data-sarcasm-clip/image/image/image"
label2id = {"Non-sarcasm": 0, "Sarcasm": 1}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
visualbert = VisualBertModel.from_pretrained("uclanlp/visualbert-vqa-coco-pre")
visualbert = visualbert.to(device)

for name, param in visualbert.named_parameters():
    if "embeddings" in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

image_encoder = models.resnet50(pretrained=True)
image_encoder.fc = nn.Identity()
image_encoder = image_encoder.to(device)
image_encoder.eval()

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class SarcasmDataset(Dataset):
    def __init__(self, json_path, img_dir):
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        caption = item["caption"]
        label = label2id[item["label"]]
        image_path = os.path.join(self.img_dir, item["image"])
        image = Image.open(image_path).convert("RGB")
        return caption, image, label

def collate_fn(batch):
    captions, images, labels = zip(*batch)
    labels = torch.tensor(labels)

    encoding = tokenizer(
        list(captions), padding=True, truncation=True, max_length=511, return_tensors="pt"
    )

    with torch.no_grad():
        img_inputs = torch.stack([image_transform(img) for img in images]).to(device)
        visual_embeds = image_encoder(img_inputs)  

    visual_embeds = visual_embeds.unsqueeze(1) 

    visual_token_type_ids = torch.ones((visual_embeds.size(0), 1), dtype=torch.long).to(device)
    visual_attention_mask = torch.ones((visual_embeds.size(0), 1), dtype=torch.long).to(device)

    # Combine attention_mask
    attention_mask = torch.cat([visual_attention_mask, encoding["attention_mask"].to(device)], dim=1)
    token_type_ids = torch.cat([visual_token_type_ids, encoding["token_type_ids"].to(device)], dim=1)

    return {
        "input_ids": torch.cat([torch.zeros_like(visual_attention_mask), encoding["input_ids"].to(device)], dim=1),
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids,
        "visual_embeds": visual_embeds,
        "labels": labels
    }

# DataLoader
train_loader = DataLoader(SarcasmDataset(train_path, img_dir), batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(SarcasmDataset(val_path, img_dir), batch_size=4, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(SarcasmDataset(test_path, img_dir), batch_size=4, shuffle=False, collate_fn=collate_fn)

class VisualBertClassifier(nn.Module):
    def __init__(self, hidden_size=768, num_labels=2):
        super().__init__()
        self.visualbert = visualbert
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids, visual_embeds):
        output = self.visualbert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            visual_embeds=visual_embeds,
            visual_token_type_ids=None,
            return_dict=True
        )
        pooled = output.pooler_output 
        return self.classifier(pooled)

model = VisualBertClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

# Training
epochs = 8
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        labels = batch["labels"].to(device)

        logits = model(**inputs)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")

# Evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        labels = batch["labels"].to(device)

        logits = model(**inputs)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        labels = batch["labels"].to(device)

        logits = model(**inputs)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="macro")
precision = precision_score(all_labels, all_preds, average="macro")
recall = recall_score(all_labels, all_preds, average="macro")

print(f"[TEST] Accuracy: {acc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
print("\nDetailed Report:")
print(classification_report(all_labels, all_preds, target_names=list(label2id.keys()), digits=4))

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1: 100%|██████████| 500/500 [01:23<00:00,  5.95it/s]


Epoch 1 Loss: 0.6594


Epoch 2: 100%|██████████| 500/500 [01:21<00:00,  6.11it/s]


Epoch 2 Loss: 0.6303


Epoch 3: 100%|██████████| 500/500 [01:22<00:00,  6.06it/s]


...

Epoch 7 Loss: 0.2353


Epoch 8: 100%|██████████| 500/500 [01:21<00:00,  6.16it/s]


Epoch 8 Loss: 0.1705


Testing: 100%|██████████| 50/50 [00:04<00:00, 11.00it/s]

[TEST] Accuracy: 0.4800 | F1: 0.4536 | Precision: 0.4640 | Recall: 0.4593

Detailed Report:
              precision    recall  f1-score   support

 Non-sarcasm     0.6422    0.5185    0.5738       135
     Sarcasm     0.2857    0.4000    0.3333        65

    accuracy                         0.4800       200
   macro avg     0.4640    0.4593    0.4536       200
weighted avg     0.5263    0.4800    0.4956       200

